In [26]:
''' beam_analysis class

#!/usr/bin/python3
# -*- coding: utf-8 -*-
'''

import numpy as np
from scipy.optimize import curve_fit
from sympy import cxxcode

class beam_analysis ():
    '''This class contains the basic methods to calculate the center of
    mass and other statistical information from a direct beam image in
    numpy array format.

    Obs.: peculiarly in this class, the results obtained in each method 
    are also returned to simplify the processing of results afterwards.

    '''
    
    def __init__ (self, beamdata, dataheader, ROIsize=250, calculate_all=False):

        # The center of mass of the whole image.
        self.data = np.float64(beamdata)
        self.Nlin, self.Ncol = self.data.shape
        #self.header = dataheader

        # Box size.
        self.ROIsize = ROIsize
        
        # Pixel size definition.
        try:
            self.PixelSize_h = float(dataheader['PixelSize_h'])
            self.PixelSize_v = float(dataheader['PixelSize_v'])
        except:
            self.PixelSize_h = 0.48  # um
            self.PixelSize_v = 0.48  # um

        # The coordinates of the data, in pixels. The Cartesian coordinates
        # are chosen for the profile, so care must be taken when dealing 
        # with the image arrays, since their vertical indices run in opposite 
        # direction and start from zero. These coordinate sets are relevant to
        # weight the center of mass etc.
        NL, NC = self.data.shape
        self.xcoords = np.arange(1, NC + 1) 
        self.ycoords = np.arange(1, NL + 1)

        # Total mass.
        self.Mass = np.sum(self.data)

        # Define centroid by flattening intensities in x and y directions. 
        # The result is usually Gaussian-like curves.
        self.beam_profile()
        self.center_of_mass()

        # Calculate all remaining observables.
        if calculate_all:
            self.center_of_mass()
            self.physical_center()
            self.FWHM()
            self.profile_cut()
            self.skewness()
            self.std_dev_raw()
            self.gaussian_fit()
            self.FWHM_gauss()
            #self.ROI_profile_center_of_mass()
            self.ROI_center_of_mass()

    #
    def beam_profile (self):
        '''Histogram profile of the beam 2D image projected onto x and y directions.'''
        self.HistProf_x = np.array([ np.sum(self.data[:, jj]) for jj in range(self.Ncol) ])
        self.HistProf_y = np.array([ np.sum(self.data[jj, :]) for jj in range(self.Nlin) ])
        #return self.HistProf_x, self.HistProf_y

    #
    def center_of_mass (self):
        '''The center of mass from the whole image, in pixel coordinates.'''
        self.CM_x = np.sum(self.xcoords * self.HistProf_x) / self.Mass
        self.CM_y = np.sum(self.ycoords * self.HistProf_y) / self.Mass
        #return self.CM_x, self.CM_y

    #
    def physical_center (self):
        '''Calculate the center in physical units.'''
        '''Given the pixel size and coordinates.'''
        self.PONI = (self.CM_x * self.PixelSize_h, self.CM_y * self.PixelSize_v)
        #return self.PONI

    #
    def profile_cut (self):
        '''Beam profile histograms as cuts through its center.'''
        def _histogram (data):
            # Return the histogram of given data, based on Sturge's number of bins.
            try:
                # Sturge's formula
                nbins = int(1 + np.log2(np.sum(data)))
            except:
                print(" (profile_cut) WARNING: histogram data is flat.")
                nbins = 1
            return np.histogram(data, bins=nbins)
        # Vertical and horizontal limits.
        HB = 0.5 * self.ROIsize    # Half-box size.
        ay, by = int(self.CM_x - HB), int(self.CM_x + HB)
        ax, bx = int(self.CM_y - HB), int(self.CM_y + HB)
        # Histograms through the beam center in vertical and
        # horizontal directions. These are cuts of the image
        # inside the box.
        self.ROIProfile_x = _histogram(self.data[ax:bx+1, int(self.CM_y)])
        self.ROIProfile_y = _histogram(self.data[int(self.CM_x), ay:by+1])

        '''Center of mass inside the ROI, from the profile, in pixels.'''
        midpoints = 0.5 * (self.ROIProfile_x[1][1:] + self.ROIProfile_x[1][:-1])
        self.ROIprofile_CM_x = ax + np.sum(midpoints * self.ROIProfile_x[0]) / np.sum(self.ROIProfile_x[0])
        midpoints = 0.5 * (self.ROIProfile_y[1][1:] + self.ROIProfile_y[1][:-1])
        self.ROIprofile_CM_y = ay + np.sum(midpoints * self.ROIProfile_y[0]) / np.sum(self.ROIProfile_y[0])
        #return self.Profile_H, self.Profile_V

    #
    def skewness (self):
        '''Skewness of the histogram.'''
        '''This function considers the 'profile' of the beam, in the 
        vertical or horizontal direction, through the beam center, as a
        distribution and evaluates its assymmetry by the calulation of
        its skewness. The unbiased (sample) Fisher's definition is
        used here, which is the standardized third momentum.

        '''
        def _skew (profile):
            # Number of bins.
            nvc = float(profile.shape[0])
            # Average value of the distribution.
            av = np.average(profile)
            # Third momentum of the distribution.
            m3 = np.average(np.array([ (pr - av)**3 for pr in profile ]))
            # Standard deviation of the sample.
            s2 = np.sum(np.array([ (pr - av)**2 for pr in profile ])) / (nvc - 1)
            # Skewness and "normalized" skewness (by interval size).
            sk = m3 / (s2**1.5)
            return sk
        #H_data, V_data = self.ROIProfile_x[0], self.ROIProfile_y[0]
        H_data, V_data = self.HistProf_x, self.HistProf_y
        self.skewness_x = _skew(H_data)
        self.skewness_y = _skew(V_data)
        #return self.skewness_x, self.skewness_y

    #
    def std_dev_raw(self):
        '''Standard deviation from raw data.'''
        SDR = lambda x, y : np.sqrt(np.sum(y * np.square(x)) / np.sum(y) -
                       (np.sum(y * x) / np.sum(y))**2)
        self.StdDevRawX = SDR(self.xcoords, self.HistProf_x)
        self.StdDevRawY = SDR(self.ycoords, self.HistProf_y)
        #return self.StdDevRawX, self.StdDevRawY

    #
    def FWHM (self):
        '''Full width at half maximum.'''
        '''Calculate FWHM from histograms of projected beam values 
        onto horizontal and vertical directions.'''
        def _fwhm (ydata):
            iA0 = np.argmax(ydata)   # Index of max value.
            A0 = ydata[iA0]          # Max value.

            # Positions (in pixels) where y < A0 /2, at left (iA1) and right (iA2).
            arr1 = np.asarray(ydata[:iA0] < 0.5 * A0).nonzero()
            iA1 = iA0 if arr1[0].shape[0] == 0 else np.max(arr1)
            #
            arr2 = np.asarray(0.5 * A0 < ydata[iA0:]).nonzero()
            iA2 = iA0 if arr2[0].shape[0] == 0 else iA0 + np.max(arr2)
            # Distance in indexes = # pixels.
            return np.abs(iA2 - iA1)
        self.FWHM_x_px = _fwhm(self.HistProf_x)
        self.FWHM_y_px = _fwhm(self.HistProf_y)
        self.FWHM_x_um = self.FWHM_x_px * self.PixelSize_h
        self.FWHM_y_um = self.FWHM_y_px * self.PixelSize_v
        #return self.FWHM_x_um, self.FWHM_y_um, self.FWHM_x_px, self.FWHM_y_px

    #
    def _gauss (self, xdata, *p0):
        '''Gaussian function.'''
        if len(p0) == 1:
            A0, mu, sigma = p0[0]
        else:
            A0, mu, sigma = p0
        return A0 * np.exp(-(xdata - mu)**2 / (2. * sigma**2))
    
    #
    def gaussian_fit (self):
        '''Fit a gaussian curve to data sets.'''
        
        '''Guess parameters: A0, mu, sigma.'''
        
        # x direction.
        p0 = [np.max(self.HistProf_x), np.argmax(self.HistProf_x), self.FWHM_x_px * 0.5]
        try:
            prm, cov = curve_fit(self._gauss, self.xcoords, self.HistProf_x, p0=p0)
            self.Gaussfit_x = {'A0_x' : prm[0],
                              'mu_x' : prm[1], 
                              'sigma_x' : prm[2],
                              'cov_x' : cov }
            self.Gaussfit_x['fitcurve_x'] = self._gauss(self.xcoords, prm)
        except Exception as err:
            print(f" (gaussian_fit) ERROR fitting x-data: {err}")
            self.Gaussfit_x = None

        # y direction.
        p0 = [np.max(self.HistProf_y), np.argmax(self.HistProf_y), self.FWHM_y_px * 0.5]
        try:
            prm, cov = curve_fit(self._gauss, self.ycoords, self.HistProf_y, p0=p0)
            self.Gaussfit_y = {'A0_y' : prm[0],
                              'mu_y' : prm[1],
                              'sigma_y' : prm[2] ,
                              'cov_y' : cov }
            self.Gaussfit_y['fitcurve_y'] = self._gauss(self.ycoords, prm)
        except Exception as err:
            print(f" (gaussian_fit) ERROR fitting y-data: {err}")
            self.Gaussfit_y = None
    
        #return self.Gaussfit_x, self.Gaussfit_y

    #
    def FWHM_gauss (self):
        '''FWHM from gaussian fitting.'''
        coef = 2.3548200450309493      # = 2. * np.sqrt(np.log(4))
        try:
            self.FWHM_gauss_x = coef * self.Gaussfit_x['sigma_x']
        except:
            self.FWHM_gauss_x = 0.
        try:
            self.FWHM_gauss_y = coef * self.Gaussfit_y['sigma_y']
        except:
            self.FWHM_gauss_y = 0.
        return

    #
    def ROI_center_of_mass (self):
        '''Center of mass of a given numpy array data, defined by the ROI size.'''

        # If formerly calculated center of mass is not well defined.
        if (self.CM_x <=  0 or self.CM_x > self.Ncol or
            self.CM_y <= 0 or self.CM_y > self.Nlin):
            self.CM_ROI_x, self.CM_ROI_y = 0., 0.
            return

        # Set coordinates of the ROI.
        coordset = np.arange(1, self.ROIsize + 1)
        HB = 0.5 * self.ROIsize

        # Narrow the array down to a box of size 'ROIsize', around the center
        # formerly calculated, and recalculate its center in pixels.
        # Box limits.
        ax0, bx0 = int(self.CM_x - HB), int(self.CM_x + HB)
        ay0, by0 = int(self.CM_y - HB), int(self.CM_y + HB)
        # Ensure the box is inside the frame.
        ax = ax0 if ax0 > 0 else 0
        bx = bx0 if bx0 < self.Ncol else 0
        #
        ay = ay0 if ay0 > 0 else 0
        by = by0 if by0 < self.Nlin else 0
        # Set ROI intervals.
        ROIdata = self.data[ax:bx, ay:by]

        # Run through elements of data and add weight intervals.
        cx = np.sum(np.array([ coordset * ROIdata[ii, :] 
                              for ii in range(self.ROIsize) ]))
        cy = np.sum(np.array([ coordset * ROIdata[:, jj] 
                              for jj in range(self.ROIsize) ]))

        # The intervals.
        self.ROImass = np.sum(ROIdata)
        self.CM_ROI_x, self.CM_ROI_y = ax + cx / self.ROImass, ay + cy / self.ROImass

        #return self.CM_ROI_x, self.CM_ROI_y



In [92]:
'''Read the hd5f data file.'''
'''And export a pandas' data frame.'''

from attr import attr
import h5py
import re
import numpy as np
import pandas as pd

#
def hdf5_read (hfname):
    '''Read hdf5 file and returns a pandas' data frame with all data.'''
    try:
        hfile = h5py.File(hfname, 'r', driver=None)

        keys, hdata = [], {}
        hfile.visit(keys.append)    # List of data.

        # Run through sets.
        for key in keys:
            # Extract indices from data name: k1 for frame number, k2 for DVF1 or DFV2.
            nkeys = re.sub('/step[_]', '', hfile[key].name)   
            nk1, nk2 = nkeys.split('_')
            k1, k2 = int(nk1), int(nk2)

            # Dictionary for one frame: slit and focus.
            if k2 == 1:
                hdata[k1] = dict()
                kword = 'slit'
            else:
                kword = 'focus'
            #
            hdata[k1][f'{kword}_img'] = np.array(hfile[key][()])
            hdata[k1][f'{kword}_analysis'] = dict(hfile[key].attrs)

    except Exception as err:
        print(f" ERROR reading file {hfname}: \n {err}")
        return None

    hdata = pd.DataFrame.from_dict(hdata, orient="columns")
    #print(f" index = {hdata.index}")
    return hdata
     
#
def show_coords (hdata, type='slit'):
    htype = f"{type}_analysis"
    for hk in hdata:
        coords = [ float(val) for k, val in hdata[hk][htype].items() ][1:]
        print(f" ({hk}) top, bottom, right, left = {coords}")

#
def main(hfname):

    hdata = hdf5_read(hfname)
    if hdata is None:
        print("ERROR: hdf5 file is empty or does not exist. Aborting.")
        return 1
    
    #print(f" <<<<< (main) hdf5 data >>>>> ")
    #show_example(hdata)
      
    return hdata

#
#if __name__ == "__main__":
#main(hfname)

wdir  = "/home/arnaldo.filho/Sirius/Carcara/slits_test/2025-05-16-Slits_check"
#filename = "dv1_images.h5"
filename = "square_scan.h5"
hfname = f"{wdir}/{filename}"

print(f"\n\n  >>>>> Reading hdf5 data... ", end="")
hdata = main(hfname)
print(" DONE.  <<<<<")




  >>>>> Reading hdf5 data...  DONE.  <<<<<


In [94]:
'''Calculate the observables for images in hdata data frame.'''

#import beam_analysis

#
def get_info (beamobj):
    '''Get all the methods of the _instantiated_ beam analysis class (object) 
    and return a dictionary with the calculated values.
    '''
    methods = [  mt for mt in beamobj.__dict__.keys() 
               if not mt.startswith('_') ]    
    #and not re.match("header", mt) ]
    return { mt : getattr(beamobj, mt) for mt in methods }

#
def observables (hdata):
    '''Calculate observables for each data set.'''

    # Run through data sets.
    for hk in hdata:

        # Extract indices from data name.
        ln = int(hk / 100)
        cl = hk - 100 * ln
        print(f"\n   >>>>> Working on set L, C = {ln:3d}, {cl:2d} ...")

        # Instantiate the class analysis for the current set 
        # and update attributes.
        dset = hdata[hk]
        for cs in ['slit', 'focus']:
            data = dset[f'{cs}_img']
            attr = dset[f'{cs}_analysis']
            # Calculate observables.
            BeAn = beam_analysis(data, attr, calculate_all=True)
            # Update attribute dictionary with new info.
            bean_info = get_info(BeAn)
            for key, val in bean_info.items():
                attr[key] = val

        print("\t done. <<<<<\n")
    return

#
#def main(hdata):
msg = '''Calculate the observables for images in hdata data frame.'''
print(f"\n ##### {msg}\n")
observables(hdata)
print("\n\n DONE. #####")
# show_data(hdata)
# return hdata

#
#if __name__ == "__main__":
#    main(hdata)




 ##### Calculate the observables for images in hdata data frame.


   >>>>> Working on set L, C =   0,  0 ...
 (gaussian_fit) ERROR fitting y-data: Optimal parameters not found: Number of calls to function has reached maxfev = 800.
	 done. <<<<<


   >>>>> Working on set L, C =   0,  1 ...
	 done. <<<<<


   >>>>> Working on set L, C =   0,  2 ...
	 done. <<<<<


   >>>>> Working on set L, C =   0,  3 ...
	 done. <<<<<


   >>>>> Working on set L, C =   0,  4 ...
 (gaussian_fit) ERROR fitting y-data: Optimal parameters not found: Number of calls to function has reached maxfev = 800.
	 done. <<<<<


   >>>>> Working on set L, C =   0,  5 ...
 (gaussian_fit) ERROR fitting x-data: Optimal parameters not found: Number of calls to function has reached maxfev = 800.
 (gaussian_fit) ERROR fitting y-data: Optimal parameters not found: Number of calls to function has reached maxfev = 800.
 (gaussian_fit) ERROR fitting x-data: Optimal parameters not found: Number of calls to function has reache

In [95]:
'''Convert pandas data frame to hdf5 and write it to file.'''

#
from matplotlib.pyplot import hist
from termcolor import ATTRIBUTES


def hdf5_write (hdata, newhfile):
    '''Write beam data from pandas data frame to hdf5 file.'''

    with h5py.File(newhfile, 'w') as hf:

        # Run through data structure.
        for hk in hdata.keys():
            dset = hdata[hk]
            # Format index.
            ln = int(hk / 100)
            cl = hk - 100 * ln
            idx = f"s_{ln:02d}_{cl:02d}" if f"{hk}".isdigit else f"s_{hk}"

            # Create a group for each data set.
            grp = hf.create_group(f"{idx}")

            # Run through data set.
            for ds, val in dset.items():

                # Create a subgroup for each type of data: image or numerical analysis.
                subgrp = grp.create_group(f"{ds}")

                # Write numpy data in data set.
                if re.search("img", ds):
                    subgrp.create_dataset(f"{ds}", data=val.astype(np.float64))
                    continue

                # Write out attributes.
                for id, att in val.items():

                    # Do not rewrite data, if it was reinserted in attributes.
                    if re.search("data", id): continue

                    if att is None:
                        # If attribute was not defined.
                        subgrp.attrs[id] = "Empty"

                    elif re.search("coord", id):
                        # Ignore x and y coordinate arrays.
                        continue

                    elif isinstance(att, np.ndarray):
                        # If attribute is an array.
                        subgrp.create_dataset(f"{id}", data=att.astype(np.float64))

                    elif re.match("ROIProfile", id):
                        # Define coordinates (midpoints) from histogram edges and write 
                        # out altogether with data.
                        histdata, edges = att
                        midpoints = 0.5 * (edges[:-1] + edges[1:])
                        histog = np.stack((midpoints, histdata), axis=1)
                        subgrp.create_dataset(f"{id}", data=histog, shape=histog.shape)
                
                    elif isinstance(att, dict):
                        # Data is the result of a Gaussian fit.
                        gauss = subgrp.create_group(f"{id}")  # Subgroup.

                        for ig, dg in att.items():
                            if re.search("fitcurve", ig):
                                # Data is fitted curve.
                                gds = gauss.create_dataset(f"gauss_{ig}", data=dg)
                            else:
                                # Data is a calculated value.
                                gauss.attrs[ig] = dg
                    else:
                        # Generic data which do not fit in a subgroup.
                        subgrp.attrs[id] = att

#
def show_data (hdata):
    '''Show a summary of the data frame hdata.'''
    print(f"##### Data type: {type(hdata)} ")
    print(f"##### Index:  \t {hdata.index}")
    print(f"##### Columns:\t {hdata.columns}")
    #print(f"####  Description: \t\n {hdata.describe()}\n\n {50 * '#'}")

    for hk, hset in hdata.items():

        # Extract indices from data name.
        ln = int(hk / 100)
        cl = hk - 100 * ln
        print(f"\n\n>>>>> Data {ln} , {cl} :")

        #hset = hdata[hk]
        for prop in ['slit', 'focus']:
            dt = f"{prop}_img"
            attr = f"{prop}_analysis"
            print(f">>>>> set {hk:4d}: type {type(hset)}")
            print(f">>>>> {dt}: {hset[dt][0,:5]} \t {type(hset[dt])} → {hset[dt].shape}")
            for idx, val in hset[attr].items():
                print(f">>> ATTR {idx} : {val} \n")
    return

#show_data(hdata)

print(f"\n\n  >>>>> Writing hdf5 data to file... ", end="")
newfile = f"{wdir}/calc_{filename}"
hdf5_write(hdata[:4], newfile)
print(" DONE.  <<<<<")




  >>>>> Writing hdf5 data to file...  DONE.  <<<<<


In [ ]:
import matplotlib.pyplot as plt

# hdata structure:
# sets: 0, 1, 2, .... 705, ... etc (= 100 * line + column scanned)
#
# columns: 'slit_img', 'slit_analysis', 'focus_img', 'focus_analysis'
#
# in each column, a dictionary with keys: 'ellapsed time (s)', 'slit_bottom', ..., 'HistProf_x', 'FWHM_y_px' etc

def get_from_table (hdata, column, item):
    hres = dict()
    for iset, hset in hdata.items():
        hres[iset] = hset.get(column)[item]
    return hres

column, item = 'focus_analysis', 'HistProf_x'
#column, item = 'focus_analysis', 'FWHM_y_um'
hres = get_from_table(hdata, column=column, item=item)

print(f"##### get from table:\n {column}    {item}")
for iset in sorted(hres.keys()):
    #print(f"      {iset:04d}   {hres[iset]:12}")
    print(f"      {iset:04d} \t    {hres[iset]}")

#plt.plot(coords, hres[0])
#plt.show()



##### get from table:
 focus_analysis    HistProf_x
      0000 	    [44. 44. 46. ... 54. 67. 74.]
      0001 	    [53. 67. 54. ... 49. 53. 61.]
      0002 	    [64. 62. 48. ... 40. 55. 50.]
      0003 	    [65. 65. 62. ... 48. 50. 52.]
      0004 	    [60. 53. 42. ... 55. 45. 53.]
      0005 	    [52. 57. 63. ... 49. 55. 56.]
      0100 	    [67. 60. 68. ... 56. 73. 51.]
      0101 	    [72. 59. 66. ... 53. 85. 61.]
      0102 	    [47. 58. 51. ... 54. 60. 44.]
      0103 	    [65. 64. 80. ... 51. 65. 50.]
      0104 	    [60. 55. 62. ... 64. 57. 62.]
      0105 	    [51. 56. 49. ... 60. 61. 49.]
      0200 	    [77. 62. 68. ... 56. 55. 59.]
      0201 	    [65. 67. 53. ... 53. 52. 57.]
      0202 	    [72. 52. 62. ... 65. 61. 54.]
      0203 	    [67. 59. 72. ... 70. 69. 75.]
      0204 	    [58. 71. 75. ... 65. 74. 62.]
      0205 	    [55. 58. 52. ... 43. 57. 77.]
      0300 	    [75. 65. 65. ... 54. 64. 57.]
      0301 	    [52. 63. 83. ... 64. 62. 59.]
      0302 	    [66. 55. 75.

In [ ]:
hdata[102]['focus_analysis']['ROIsize']


250